In [ ]:
import pandas as pd
import sqlite3 as sql

# ---------- Merging and Joining ----------
# 1.
with sql.connect('chinook.db') as connection:
    df_customers = pd.read_sql_query('SELECT * FROM customers', con=connection)
    df_invoices = pd.read_sql_query('SELECT * FROM invoices', con=connection)

result = pd.merge(df_customers, df_invoices, on='CustomerId', how='inner')
print(result.shape)

invoices_per_customer = (
    result
    .groupby(['CustomerId', 'FirstName', 'LastName'])
    .size()
    .rename('InvoiceCount')
    .reset_index()
)
display(invoices_per_customer.head(10))
print(invoices_per_customer.shape)

# Total amount over all invoices
print(result['Total'].sum())

In [ ]:
# ---------- Merging and Joining ----------
# 2. Left and outer joins
df_movie = pd.read_csv('movie.csv')
df_color = df_movie[['director_name', 'color']].dropna()
df_reviews = df_movie[['director_name', 'num_critic_for_reviews']].dropna()

left_result = pd.merge(df_color, df_reviews, on='director_name', how='left')
full_outer_result = pd.merge(df_color, df_reviews, on='director_name', how='outer')

display(left_result.sample(10))
print(left_result.shape)
display(full_outer_result.sample(10))
print(full_outer_result.shape)

In [ ]:
# ---------- Grouping and Aggregating ----------
# 1. Titanic: mean age (float), total fare, passenger count per class
df_excel = pd.read_excel('titanic.xlsx')

new_df = df_excel.groupby('Pclass').agg(
    Age_mean=('Age', 'mean'),
    Fare_sum=('Fare', 'sum'),
    Passenger_count=('PassengerId', 'count'),
).reset_index()

display(new_df)

In [ ]:
# ---------- Grouping and Aggregating ----------
# 2. Movies: total critic reviews and average duration per color/director
grouped_movie = df_movie.groupby(['color', 'director_name']).agg(
    total_critic_reviews=('num_critic_for_reviews', 'sum'),
    average_duration=('duration', 'mean'),
).reset_index()

display(grouped_movie.sample(10))
print(grouped_movie.shape)

In [ ]:
# ---------- Grouping and Aggregating ----------
# 3. Flights: nested grouping by Year and Month
df_parquit = pd.read_parquet('./drive/MyDrive/data/flights/')

# Convert columns to numeric, coercing errors to NaN
df_parquit['ArrDelay'] = pd.to_numeric(df_parquit['ArrDelay'], errors='coerce')
df_parquit['DepDelay'] = pd.to_numeric(df_parquit['DepDelay'], errors='coerce')

result = df_parquit.groupby(['Year', 'Month']).agg(
    Total_Flights=('Month', 'size'),        # number of flights per month
    Avg_ArrDelay=('ArrDelay', 'mean'),      # average arrival delay
    Max_DepDelay=('DepDelay', 'max'),       # maximum departure delay
).reset_index()

display(result)
print(result.shape)

In [ ]:
# ---------- Applying Functions ----------
# 1. Titanic: Age group
df_excel = pd.read_excel('titanic.xlsx')

def modify_age(row):
    return "Child" if row['Age'] < 18 else "Adult"

df_excel['Age_group'] = df_excel.apply(modify_age, axis=1)
display(df_excel.sample(10))

In [ ]:
# ---------- Applying Functions ----------
# 2. Employees: normalize BASE_SALARY within each department
df_employee = pd.read_csv('employee.csv')

def min_max(x):
    rng = x.max() - x.min()
    if rng == 0:                      # all salaries equal in this department
        return x * 0.0
    return (x - x.min()) / rng

df_employee['normalized_salary'] = (
    df_employee.groupby('DEPARTMENT')['BASE_SALARY'].transform(min_max)
)

display(df_employee[['DEPARTMENT', 'BASE_SALARY', 'normalized_salary']].sample(10))
print(df_employee.shape)

In [ ]:
# 3. Movies: classify duration
def classify_duration(duration):
    if pd.isna(duration):
        return 'Unknown'
    elif duration < 60:
        return 'Short'
    elif duration <= 120:
        return 'Medium'
    else:
        return 'Long'

df_movie['duration_category'] = df_movie['duration'].apply(classify_duration)
display(df_movie[['movie_title', 'duration', 'duration_category']].head(10))

In [ ]:
# ---------- Using pipe ----------
# 1. Titanic pipeline
df_titanic = pd.read_excel('titanic.xlsx')
df_titanic['Age'] = df_titanic['Age'].fillna(df_titanic['Age'].mean())

def filter_by_survival(df):
    return df[df['Survived'] == 1]

def add_fare_per_age(df, col_name):
    return df.assign(**{col_name: df['Fare'] / df['Age']})

df_cleaned = (
    df_titanic
    .pipe(add_fare_per_age, 'Fare_Per_Age')
    .pipe(filter_by_survival)
)
display(df_cleaned.head())
print(df_cleaned.shape)

In [ ]:
# 2. Flights pipeline: DepDelay > 30, then Delay_Per_Hour = DepDelay / ScheduledDuration
# NOTE: change DURATION_COL if your parquet uses another name for the scheduled duration.
DURATION_COL = 'ScheduledDuration'

def filter_delayed(df, threshold=30):
    return df[df['DepDelay'] > threshold]

def add_delay_per_hour(df, duration_col=DURATION_COL):
    return df.assign(Delay_Per_Hour=df['DepDelay'] / df[duration_col])

df_flights_pipeline = (
    df_parquit
    .pipe(filter_delayed)
    .pipe(add_delay_per_hour)
)

print(df_parquit.shape, '->', df_flights_pipeline.shape)
display(df_flights_pipeline[['DepDelay', DURATION_COL, 'Delay_Per_Hour']].head())